## Questão 6 - Dimensão de calendário

### Cenário

O Sr. Almir quer saber. 'Qual é o dia da semana (Segunda, Terça ... ) que temos a pior média de vendas?' para decidir se vale a pena fechar a loja nesses dias. 

Um estagiário fez um GROUP BY dia_semana direto na tabela de vendas e disse que a Terça-feira era ótima, com média de R$ 5.000,00. 

O problema: O estagiário esqueceu que em muitas terças-feiras a loja abriu mas vendeu zero. Como esses dias nao existem na tabela de vendas (`vendas_2023_2024.csv`), eles foram ignorados no cálculo da média, inflando o resultado. Precisamos corrigir isso utilizando um calendario de datas (dimensão de datas)

### Premissas obrigatórias:

- O periodo de analise deve considerar todas as datas entre a menor e a maior data_venda presentes no arquivo.
- A loja esteve aberta em todos os dias do periodo (inclusive fins de semana).
- Dias sem registro na tabela de vendas devem ser considerados como `valor_venda = 0`.
- “Vendas diarias" correspondem à soma de valor_venda por dia.
- A media de vendas por dia da semana deve considerar todos os dias do calendario, inclusive
os dias sem venda.
- O nome do dia da semana deve ser apresentado em portugués (Segunda-feira,
Terça-feira, etc.).

### Tarefa:

1. Construa uma dimensão de datas utilizando sql 
2. Cruze a dimensão de datas com a tabela de vendas para análise (nao esqueça de considerar os dias sem vendas).

In [1]:
import pandas as pd

In [2]:
vendas = pd.read_csv('../datasets_tratados/vendas_normalizadas.csv')
vendas['sale_date'] = pd.to_datetime(vendas['sale_date'])

vendas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9895 entries, 0 to 9894
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   id          9895 non-null   int64         
 1   id_client   9895 non-null   int64         
 2   id_product  9895 non-null   int64         
 3   qtd         9895 non-null   int64         
 4   total       9895 non-null   float64       
 5   sale_date   9895 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(4)
memory usage: 464.0 KB


> 1. Dimensão de datas 

In [3]:
# Definir o período baseado na base de vendas
data_inicio = vendas['sale_date'].min()
data_fim = vendas['sale_date'].max()

# Criar a sequência de datas
datas = pd.date_range(start=data_inicio, end=data_fim, freq='D')

# Criar dimensão calendário
df_calendario = pd.DataFrame(datas, columns=['data'])

# Mapeamento dos dias da semana em português
dias_semana_pt = {
    0: 'Segunda-feira', 
    1: 'Terça-feira', 
    2: 'Quarta-feira', 
    3: 'Quinta-feira', 
    4: 'Sexta-feira', 
    5: 'Sábado', 
    6: 'Domingo'
}

# Extrair o dia da semana e mapear
df_calendario['dia_semana'] = df_calendario['data'].dt.dayofweek.map(dias_semana_pt)

# Agrupar as vendas por dia antes do merge
vendas_diarias = vendas.groupby('sale_date')['total'].sum().reset_index()

vendas_diarias.head()

,sale_date,total
0,2023-01-01,4207939.25
1,2023-01-02,5102799.80
2,2023-01-03,8299127.05
3,2023-01-04,4707759.40
4,2023-01-05,5525706.90


In [4]:
# Define como vai ser exibido os números float de forma global
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

> 2. Cruze a dimensão de datas com a tabela de vendas para análise (nao esqueça de considerar os dias sem vendas)

In [5]:
# Cruzamento final para análise
# Garante que dias com zero vendas apareçam
df_final = pd.merge(df_calendario, vendas_diarias, left_on='data', right_on='sale_date', how='left')

# Preencher valores vazios com 0
df_final['total'] = df_final['total'].fillna(0)

# RESULTADO: Média real por dia da semana
resultado = df_final.groupby('dia_semana')['total'].mean().sort_values()
print(resultado)

dia_semana
Domingo         3,319,503.57
Segunda-feira   3,465,137.71
Quarta-feira    3,535,265.63
Quinta-feira    3,626,232.44
Terça-feira     3,627,045.76
Sábado          3,710,540.55
Sexta-feira     3,715,003.41
Name: total, dtype: float64


### **Análise: Média Real de Vendas por Dia da Semana**

* **Problema Detetado**: A análise anterior ignorava os dias em que a loja abria mas não realizava vendas, o que gerava médias de faturamento artificialmente altas.
* **Correção Aplicada**: Utilizou-se uma **Dimensão Calendário** para abranger todas as datas entre a primeira e a última venda. Dias sem registo de transações foram contabilizados com o valor de **R$ 0,00** para refletir a ociosidade real.
* **Resultado da Média Real**:
    * **Pior Desempenho**: O **Domingo** apresenta a pior média real de vendas de toda a semana.
    * **Impacto**: Ao incluir os dias com "venda zero", a média real caiu significativamente em comparação com a estimativa inicial, que era baseada apenas em dias com sucesso de venda.

**Conclusão para o Sr. Almir**:
A loja apresenta uma subutilização crítica aos **Domingo**. Com base nestes dados estatísticos reais, existe fundamentação para avaliar o encerramento da unidade neste dia específico ou a implementação de ações promocionais para cobrir os custos fixos de manutenção da loja aberta.